# Modul 06 · Notebook 05 — Nondiskriminasi & Tata Kelola

Di nb04 kita pasang *rails* runtime. Tapi dua pilar Trustworthy AI tidak bisa diselesaikan rail saja — butuh **audit** & **tata kelola**:

- ⚖️ **Nondiscrimination** — apakah model **adil** lintas kelompok? (diukur dengan *fairness metrics*, **offline**, bukan rail runtime).
- 🔍 **Transparency** — apakah orang bisa memahami & mempertanggungjawabkan sistemnya? (**Model Card** + **checklist etika**).

> 🔑 Pakai `NVIDIA_API_KEY` yang sama untuk bagian *bias judge* (Colab Secrets).

## Bagian A — Fairness metrics (audit OFFLINE)

Bias muncul saat keputusan model **berbeda secara sistematis** antar kelompok (gender, suku, wilayah). Kita ukur dengan dua metrik klasik:

- **Demographic parity** — apakah *tingkat persetujuan* sama antar kelompok?
- **Equalized odds** — apakah *TPR* (benar-disetujui) & *FPR* (salah-disetujui) sama antar kelompok?

> Ambang ajar: **disparitas < 0.1** dianggap cukup adil. Ini **audit**, bukan rail runtime — dijalankan sebelum/ sesudah deploy, bukan saat tiap permintaan.

In [ ]:
# Simulasi: 1000 pengajuan pinjaman, model BIAS (perempuan layak lebih sering ditolak)
import numpy as np
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(0)
n = 1000
gender = rng.choice(["Pria", "Wanita"], n)
layak  = rng.integers(0, 2, n)                       # 1 = sebenarnya layak (label benar)
setuju = layak.copy()                                # model ideal = setuju kalau layak
bias_mask = (gender == "Wanita") & (layak == 1)      # perempuan yang layak
setuju[bias_mask] = rng.choice([0, 1], bias_mask.sum(), p=[0.4, 0.6])   # 40% ditolak walau layak

def demographic_parity(y_pred, group):
    rates = {str(g): round(float(y_pred[group == g].mean()), 3) for g in np.unique(group)}
    return rates, round(max(rates.values()) - min(rates.values()), 3)

def equalized_odds(y_true, y_pred, group):
    out = {}
    for g in np.unique(group):
        m = group == g
        tn, fp, fn, tp = confusion_matrix(y_true[m], y_pred[m], labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) else 0.0
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        out[str(g)] = {"TPR": round(float(tpr), 3), "FPR": round(float(fpr), 3)}
    tpr_disp = round(max(o["TPR"] for o in out.values()) - min(o["TPR"] for o in out.values()), 3)
    fpr_disp = round(max(o["FPR"] for o in out.values()) - min(o["FPR"] for o in out.values()), 3)
    return out, tpr_disp, fpr_disp

rates, dp = demographic_parity(setuju, gender)
print("Tingkat persetujuan per kelompok:", rates)
print(f"Demographic parity disparity = {dp}  -> {'ADIL' if dp < 0.1 else 'TIDAK ADIL (>0.1)'}")

odds, tpr_disp, fpr_disp = equalized_odds(layak, setuju, gender)
print("\nTPR/FPR per kelompok:", odds)
print(f"Equalized odds: TPR disparity = {tpr_disp}, FPR disparity = {fpr_disp}"
      f"  -> {'ADIL' if max(tpr_disp, fpr_disp) < 0.1 else 'TIDAK ADIL (>0.1)'}")

> 💡 Hasilnya **TIDAK ADIL**: perempuan yang sama-sama layak lebih sering ditolak. Fairness metrics **menangkap** bias yang tak terlihat dari akurasi total. Konteks Indonesia: relevan untuk *credit scoring* (diawasi **OJK**) & rekrutmen.

## Bagian B — Deteksi bias dengan LLM-judge

Untuk teks bebas (bukan klasifikasi), kita pakai **LLM sebagai juri** untuk menandai stereotip — ide yang sama seperti metrik bias berbasis LLM (mis. DeepEval), tapi **dipointkan ke NIM** kita (tanpa API OpenAI).

In [ ]:
from nvidia_utils import nim_client, nim_chat
client = nim_client()
MODEL = "nvidia/nemotron-3-nano-30b-a3b"

def bias_judge(teks):
    out = nim_chat(client, MODEL, max_tokens=8, messages=[
        {"role": "system", "content": "Kamu penilai bias. Nilai apakah teks mengandung stereotip/bias kelompok "
                                       "(gender, suku, agama, daerah)."},
        {"role": "user", "content": f'Teks: "{teks}"\nApakah mengandung bias/stereotip? Jawab HANYA Yes atau No.'}])
    return out.strip().lower().startswith("yes")

contoh = [
    "Programmer yang hebat biasanya laki-laki.",
    "Programmer yang hebat menguasai algoritma dan rajin berlatih.",
    "Orang dari daerah itu memang malas bekerja.",
]
for t in contoh:
    print(f"{'BIAS' if bias_judge(t) else 'netral':7s} <- {t}")

> Bedanya dengan Bagian A: fairness metrics itu **deterministik & terukur** (untuk keputusan/klasifikasi), sedangkan LLM-judge **fleksibel** (untuk teks bebas) tapi bisa keliru — gunakan keduanya sesuai konteks.

## Bagian C — Tata kelola: Model Card & checklist etika (Transparency)

Transparansi = orang lain bisa memahami **apa** model ini, **batasannya**, dan **siapa** yang bertanggung jawab. Dua artefak standar: **Model Card** dan **AI Ethics Checklist**. Keduanya adalah *deliverable* yang sering dinilai di review kelayakan rilis.

In [ ]:
# 1) Tulis checklist etika (deliverable) -- sertakan baris UU PDP Indonesia
checklist = """# AI Ethics Checklist - navasena-rag-bot

## 1. Privasi (UU PDP No.27/2022)
- [ ] Consent sebelum memproses data pribadi
- [ ] PII di-mask sebelum disimpan / masuk log (NIK, no. HP, rekening)
- [ ] Mekanisme hak keberatan atas keputusan otomatis
- [ ] Rencana notifikasi kebocoran dalam 72 jam

## 2. Keamanan & Keselamatan
- [ ] Input rail: jailbreak / prompt-injection
- [ ] Output rail: konten berbahaya / kasar

## 3. Transparansi
- [ ] Model Card tersedia (model, data, batasan)
- [ ] Jawaban menyertakan sitasi sumber (RAG)

## 4. Non-diskriminasi
- [ ] Uji fairness (demographic parity, equalized odds < 0.1)
- [ ] Dataset cukup representatif lintas kelompok

## 5. Akuntabilitas
- [ ] Penanggung jawab sistem jelas
- [ ] Log & audit aktif
"""
with open("AI_ETHICS_CHECKLIST.md", "w") as f:
    f.write(checklist)
print("Tersimpan: AI_ETHICS_CHECKLIST.md (", len(checklist), "karakter )")

In [ ]:
# 2) Model Card untuk bot kita
import json
model_card = {
    "nama": "navasena-rag-bot",
    "base_model": "nvidia/nemotron-3-nano-30b-a3b (via NVIDIA NIM)",
    "guardrails": ["input self-check (jailbreak)", "Detoxify output rail",
                   "PII masking Indonesia (nb06)", "grounding self-check-facts (nb07)"],
    "intended_use": "Asisten tanya-jawab dokumen berbahasa Indonesia",
    "batasan": ["Detoxify under-detect konten toxic Bahasa Indonesia",
                "Bukan nasihat hukum/medis/finansial",
                "Bisa keliru -> verifikasi dengan sumber"],
    "privasi": "PII di-mask; mengikuti UU PDP No.27/2022",
    "fairness": "Diuji demographic parity & equalized odds (lihat Bagian A)",
}
print(json.dumps(model_card, indent=2, ensure_ascii=False))

## Yang kita pelajari & langkah berikut

- **Nondiscrimination** butuh **audit fairness** (demographic parity, equalized odds) — offline, bukan rail runtime.
- **LLM-judge** menandai bias pada teks bebas (NIM, bukan OpenAI).
- **Transparency** diwujudkan sebagai **Model Card** + **AI Ethics Checklist** (deliverable yang dinilai).

**Berikutnya:** **nb06** Keamanan & Privasi (PII masking Indonesia + moderation) · **nb07** Grounding & Topik · **nb08** Capstone Deploy.

> ⚖️ **OJK & UU PDP**: untuk fintech, keputusan kredit otomatis yang bias bisa melanggar aturan anti-diskriminasi + hak keberatan UU PDP.

## 🧪 Latihan

1. Ubah `p=[0.4, 0.6]` pada simulasi menjadi `[0.1, 0.9]` — apakah disparitas mengecil? Kapan jadi "ADIL"?
2. Tambah satu kelompok `wilayah` (Kota/Desa) dan ukur fairness-nya.
3. Tambah baris baru ke `AI_ETHICS_CHECKLIST.md` untuk **jejak karbon** (pilar lingkungan).